# HFP vs GLA v2 — LM Baseline + Uzun-Bağlam Karar Deneyi (WikiText-2)

**v2 düzeltmeleri:** (1) GLA'ya çıkış LayerNorm'u eklendi (v1'de LM ölçeğinde NaN'a uçuyordu). (2) Görev B yeniden tasarlandı: seq 1024'te *eğitim* training-length cliff'e takılıyor (RESULTS §3) — doğrusu **train-short → infer-long**: seq 256'da eğit, aynı ağırlıkları 256/1024/2048'de değerlendir. (3) Analiz NaN'a dayanıklı.

**Görev A — GLA baseline (seq 256):** LR taraması seed 0 → en iyi LR × 3 seed. HFP referansları ablasyondan.

**Görev B — uzun-bağlam yazım-kuralı kararı:** `cubic+additive+dpfp` vs `cubic+delta+dpfp` vs GLA; seq 256'da eğit, {256, 1024, 2048}'de PPL ölç.

**ÖNCEDEN yazılı kriterler:**
1. HFP-best, GLA'nın −2 SE bandında veya üstündeyse → aile-standardı DOĞRULANDI.
2. **En uzun eval'de (2048)** delta, additive'i 3-seed ort. >2 SE geçerse → 'delta uzun bağlamda' DOĞRULANDI; aksi halde reçete additive.

> Platform: Kaggle T4/P100 veya Colab. GLA LR taraması + 3 seed + 9 Görev-B koşusu ≈ ablasyon süresine yakın.

In [ ]:
# --- KURULUM ---
import os, subprocess, sys

if os.path.exists('/content'):
    BASE = '/content'; print('Google Colab ortami')
elif os.path.exists('/kaggle'):
    BASE = '/kaggle/working'; print('Kaggle ortami')
else:
    BASE = '.'; print('Yerel ortam')

REPO = os.path.join(BASE, 'HFP')
if not os.path.isdir(REPO):
    subprocess.run(['git', 'clone', 'https://github.com/kayra-hn/HFP.git', REPO], check=True)
else:
    subprocess.run(['git', '-C', REPO, 'pull'], check=True)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'transformers>=4.40'], check=True)

os.chdir(REPO)
sys.path.insert(0, REPO)

import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()} - {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "N/A"}')
print('Kurulum tamam!')

In [ ]:
# --- WRAPPER SCRIPT OLUSTUR (GLA + HFP, tek protokol) ---
import os
WRAPPER = os.path.join(REPO, '_bench_train.py')

script = r'''
import argparse, os, sys, math, csv, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import urllib.request
from transformers import AutoTokenizer, get_cosine_schedule_with_warmup


# ============ GLA (Yang ve ark. 2023 ailesi) — chunkwise-paralel LM ============
# baseline_compare.py'deki GLA'nin LM-olcegine tasinmis hali. Veri-bagimli
# per-kanal unutma kapisi a_t = sigmoid(W_a x_t); S_t = diag(a_t) S_{t-1} + k_t v_t^T.
# Kapi girdiden bagimsiz-oz-yinelemeli oldugundan (lam onceden bilinir) chunkwise
# TAM paralel cozulur (log-uzayda kumulatif toplam; hfp_bulk_state.py exp-yoluyla
# ayni cebir). Kasitli farklar (adil dis referans icin): pencereli-attention YOK,
# feature-map elu+1, normalizer yok (saf GLA okumasi o_t = q_t S_t).
class GLALayer(nn.Module):
    def __init__(self, hid, ffn_mult=4, rec_block=16):
        super().__init__()
        self.hid = hid; self.rec_block = rec_block
        self.q = nn.Linear(hid, hid, bias=False)
        self.k = nn.Linear(hid, hid, bias=False)
        self.v = nn.Linear(hid, hid, bias=False)
        self.a = nn.Linear(hid, hid)
        nn.init.constant_(self.a.bias, 2.0)      # sigmoid(2)~0.88 uzun-ufuk baslangici
        self.o = nn.Linear(hid, hid, bias=False)
        self.out_norm = nn.LayerNorm(hid)   # [v2] cikis normu: paydasiz okuma LM'de patliyordu (NaN)
        self.norm1 = nn.LayerNorm(hid)
        self.norm2 = nn.LayerNorm(hid)
        self.ffn = nn.Sequential(nn.Linear(hid, ffn_mult * hid), nn.GELU(),
                                 nn.Linear(ffn_mult * hid, hid))

    def forward(self, x):
        B, L, H = x.shape
        dt = x.dtype; dev = x.device
        q = F.elu(self.q(x)) + 1.0
        k = F.elu(self.k(x)) + 1.0
        v = self.v(x)
        loga = F.logsigmoid(self.a(x))            # (B,L,H) log-kapilar
        S = x.new_zeros(B, H, H)
        outs = []
        for s0 in range(0, L, self.rec_block):
            qb = q[:, s0:s0 + self.rec_block]
            kb = k[:, s0:s0 + self.rec_block]
            vb = v[:, s0:s0 + self.rec_block]
            m = qb.size(1)
            cs = torch.cumsum(loga[:, s0:s0 + m], dim=1)      # (B,m,H) log A_i
            A = torch.exp(cs)
            # cross-blok: eski S katkisi A_i ile soner
            out_cross = torch.bmm(qb * A, S)                   # (B,m,H)
            # intra-blok: pair (i,j<=i) katsayisi exp(cs_i - cs_j)
            ii = torch.arange(m, device=dev).view(m, 1)
            jj = torch.arange(m, device=dev).view(1, m)
            causal = (ii >= jj).to(dt)
            Dm = torch.exp(cs.unsqueeze(2) - cs.unsqueeze(1)) * causal.view(1, m, m, 1)
            Sq = torch.einsum('bih,bijh,bjh->bij', qb, Dm, kb)
            outs.append(out_cross + torch.bmm(Sq, vb))
            # durum guncelle (blok sonu)
            A_m = A[:, -1]
            K_rev = kb * torch.exp(cs[:, -1:] - cs)
            S = S * A_m.unsqueeze(-1) + torch.bmm(K_rev.transpose(1, 2), vb)
        o = self.out_norm(torch.cat(outs, dim=1))   # [v2]
        x = self.norm1(x + self.o(o))
        x = self.norm2(x + self.ffn(x))
        return x


class GLAForCausalLM(nn.Module):
    def __init__(self, vocab, hid=256, layers=4, max_pos=2048, rec_block=16):
        super().__init__()
        self.emb = nn.Embedding(vocab, hid)
        pe = torch.zeros(max_pos, hid)
        pos = torch.arange(max_pos).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, hid, 2).float() * (-math.log(10000.0) / hid))
        pe[:, 0::2] = torch.sin(pos * div); pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe * 0.3)      # pe_scale=0.3, HFP ile ayni denge
        self.layers = nn.ModuleList([GLALayer(hid, rec_block=rec_block) for _ in range(layers)])
        self.norm = nn.LayerNorm(hid)
        self.head = nn.Linear(hid, vocab, bias=False)
        self.head.weight = self.emb.weight         # weight tying (HFP ile ayni)

    def forward(self, idx, labels=None):
        x = self.emb(idx) * math.sqrt(self.emb.weight.size(1)) + self.pe[:idx.size(1)]
        for lyr in self.layers:
            x = lyr(x)
        logits = self.head(self.norm(x))
        loss = None
        if labels is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), labels.view(-1))
        class Out: pass
        out = Out(); out.logits = logits; out.loss = loss
        return out


def estimate_loss(model, train_data, val_data, eval_iters, seq_length, batch_size, device):
    out = {}
    model.eval()
    for split, data in [('train', train_data), ('val', val_data)]:
        losses = []
        for _ in range(eval_iters):
            ix = torch.randint(len(data) - seq_length, (batch_size,))
            x = torch.stack([torch.from_numpy(data[i:i+seq_length].astype(np.int64)) for i in ix]).to(device)
            y = torch.stack([torch.from_numpy(data[i+1:i+seq_length+1].astype(np.int64)) for i in ix]).to(device)
            with torch.no_grad():
                output = model(x, labels=y)
            losses.append(output.loss.item())
        out[split] = np.mean(losses)
    model.train()
    return out


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('--model', type=str, required=True, choices=['gla', 'hfp'])
    parser.add_argument('--decay_mode', type=str, default='cubic_flux_chunked')
    parser.add_argument('--write_rule', type=str, default='additive')
    parser.add_argument('--key_feature_map', type=str, default='dpfp')
    parser.add_argument('--seed', type=int, required=True)
    parser.add_argument('--lr', type=float, default=5e-4)
    parser.add_argument('--max_iters', type=int, default=5000)
    parser.add_argument('--seq_length', type=int, default=256)
    parser.add_argument('--batch_size', type=int, default=16)
    parser.add_argument('--patience', type=int, default=7)
    parser.add_argument('--eval_interval', type=int, default=200)
    parser.add_argument('--warmup_steps', type=int, default=100)
    parser.add_argument('--log_tag', type=str, required=True)
    parser.add_argument('--eval_lens', type=str, default='')   # orn '256,1024,2048'
    args = parser.parse_args()

    torch.manual_seed(args.seed); np.random.seed(args.seed); random.seed(args.seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(args.seed)

    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    tokenizer = AutoTokenizer.from_pretrained('gpt2')

    # --- Veri: colab_lm_ablation.ipynb ile BIREBIR ayni ---
    print('WikiText-2 verisi indiriliyor (Direct URL)...', flush=True)
    if not os.path.exists('wiki_train.txt'):
        urllib.request.urlretrieve('https://raw.githubusercontent.com/pytorch/examples/master/word_language_model/data/wikitext-2/train.txt', 'wiki_train.txt')
    if not os.path.exists('wiki_valid.txt'):
        urllib.request.urlretrieve('https://raw.githubusercontent.com/pytorch/examples/master/word_language_model/data/wikitext-2/valid.txt', 'wiki_valid.txt')
    with open('wiki_train.txt', 'r', encoding='utf-8') as f:
        train_text = f.read()
    with open('wiki_valid.txt', 'r', encoding='utf-8') as f:
        val_text = f.read()
    chunk_size = 100000
    train_list = []
    for i in range(0, len(train_text), chunk_size):
        train_list.extend(tokenizer.encode(train_text[i:i+chunk_size], truncation=False))
    train_data = np.array(train_list, dtype=np.int64)
    val_list = []
    for i in range(0, len(val_text), chunk_size):
        val_list.extend(tokenizer.encode(val_text[i:i+chunk_size], truncation=False))
    val_data = np.array(val_list, dtype=np.int64)
    print('Train: {:,} | Val: {:,} tokens'.format(len(train_data), len(val_data)), flush=True)

    vocab_size = len(tokenizer)
    if args.model == 'gla':
        _maxlen = max([args.seq_length] + [int(s) for s in args.eval_lens.split(',') if s])
        model = GLAForCausalLM(vocab_size, hid=256, layers=4,
                               max_pos=_maxlen + 8).to(device)
    else:
        from hfp.models.configuration_hfp import HFPConfig
        from hfp.models.modeling_hfp import HFPForCausalLM
        config = HFPConfig(
            vocab_size=vocab_size, hidden_size=256, num_hidden_layers=4,
            num_attention_heads=4,
            max_position_embeddings=max([args.seq_length] + [int(s) for s in args.eval_lens.split(',') if s]),
            short_len=8, bulk_dim=32,
            decay_mode=args.decay_mode,
            key_feature_map=args.key_feature_map,
            write_rule=args.write_rule,
            ffn_type='standard',
            rec_block=16)
        model = HFPForCausalLM(config).to(device)

    opt = torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=0.1)
    sch = get_cosine_schedule_with_warmup(opt, args.warmup_steps, args.max_iters)

    print('Params: {:.2f}M'.format(sum(p.numel() for p in model.parameters()) / 1e6), flush=True)
    print('Model: {} (decay={} write={} fmap={})'.format(
        args.model, args.decay_mode, args.write_rule, args.key_feature_map), flush=True)

    best_val_loss = float('inf')
    patience_ctr = 0
    log_file = '{}_log.csv'.format(args.log_tag)
    with open(log_file, 'w', newline='') as lf:
        csv.writer(lf).writerow(['step', 'train_loss', 'val_loss', 'val_ppl'])

    for it in range(args.max_iters):
        if it % args.eval_interval == 0 or it == args.max_iters - 1:
            losses = estimate_loss(model, train_data, val_data, 50, args.seq_length, args.batch_size, device)
            tl = losses['train']; vl = losses['val']; val_ppl = math.exp(vl)
            print('Step {}: train {:.4f} val {:.4f} ppl {:.2f}'.format(it, tl, vl, val_ppl), flush=True)
            with open(log_file, 'a', newline='') as lf:
                csv.writer(lf).writerow([it, tl, vl, val_ppl])
            if vl < best_val_loss:
                best_val_loss = vl; patience_ctr = 0
            else:
                patience_ctr += 1
                if patience_ctr >= args.patience:
                    print('Early stop @ step {}. Best val: {:.4f}'.format(it, best_val_loss), flush=True)
                    break
        ix = torch.randint(len(train_data) - args.seq_length, (args.batch_size,))
        x = torch.stack([torch.from_numpy(train_data[i:i+args.seq_length].astype(np.int64)) for i in ix]).to(device)
        y = torch.stack([torch.from_numpy(train_data[i+1:i+args.seq_length+1].astype(np.int64)) for i in ix]).to(device)
        output = model(x, labels=y)
        if not torch.isfinite(output.loss):
            print('DIVERGED (NaN/Inf) @ step {} — kosu sonlandirildi.'.format(it), flush=True)
            with open(log_file, 'a', newline='') as lf:
                csv.writer(lf).writerow([it, 'nan', 'nan', 'nan'])
            break
        opt.zero_grad(set_to_none=True)
        output.loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step(); sch.step()

    print('DONE best_val_loss={:.4f} best_ppl={:.2f}'.format(best_val_loss, math.exp(best_val_loss)), flush=True)

    # [v2] train-short -> infer-long: ayni agirliklari farkli uzunluklarda degerlendir
    if args.eval_lens:
        lens = [int(s) for s in args.eval_lens.split(',')]
        with open('{}_lens.csv'.format(args.log_tag), 'w', newline='') as lf:
            wcsv = csv.writer(lf); wcsv.writerow(['eval_len', 'val_loss', 'val_ppl'])
            for elen in lens:
                bs = max(1, (args.batch_size * args.seq_length) // elen)
                vl = estimate_loss(model, train_data, val_data, 30, elen, bs, device)['val']
                wcsv.writerow([elen, vl, math.exp(vl)])
                print('EVAL_LEN {}: val {:.4f} ppl {:.2f}'.format(elen, vl, math.exp(vl)), flush=True)

if __name__ == '__main__':
    main()
'''.strip()

with open(WRAPPER, 'w') as f:
    f.write(script)
print(f'Wrapper yazildi: {WRAPPER}')

## Görev A — GLA baseline, seq 256

GLA için LR taraması seed 0'da ({3e-4, 5e-4, 1e-3}), sonra en iyi LR ile seed 1 ve 2 (toplam 5 koşu). HFP seq-256 sonuçları ablasyondan referans alınır — yeniden koşulmaz. (Metodoloji kuralı: mode-başına LR taraması; sabit-LR kıyası haksızdır.)

In [ ]:
# --- GOREV A: GLA LR taramasi (seed 0) + en iyi LR x seed 1,2 ---
import subprocess, sys, time, csv, math, torch
assert torch.cuda.is_available(), 'GPU SECILI DEGIL — once acceleratoru ayarlayin, sonra kosun.'

def run(cmd_args, tag):
    print(f'\n>>> {tag} basliyor...', flush=True)
    t0 = time.time()
    r = subprocess.run([sys.executable, WRAPPER] + cmd_args)
    print(f'<<< {tag} bitti ({(time.time()-t0)/60:.1f} dk, rc={r.returncode})', flush=True)

def best_val(tag):
    try:
        with open(f'{tag}_log.csv') as f:
            rows = list(csv.DictReader(f))
        vals = [float(r['val_loss']) for r in rows
                if r['val_loss'] != 'nan' and not math.isnan(float(r['val_loss']))]
        return min(vals) if vals else float('inf')   # NaN'li kosu LR yarisini kazanamaz
    except Exception:
        return float('inf')

# 1) LR taramasi (seed 0)
LRS = ['3e-4', '5e-4', '1e-3']
for lr in LRS:
    tag = f'gla_s0_lr{lr}'
    run(['--model', 'gla', '--seed', '0', '--lr', lr, '--seq_length', '256',
         '--batch_size', '16', '--max_iters', '5000', '--log_tag', tag], tag)

sweep = {lr: best_val(f'gla_s0_lr{lr}') for lr in LRS}
BEST_LR = min(sweep, key=sweep.get)
assert sweep[BEST_LR] != float('inf'), 'TUM LRler diverge etti — GLA implementasyonu incelenmeli, seedlere GECME.'
print(f'\nLR taramasi: {sweep}')
print(f'En iyi GLA LR: {BEST_LR}')

# 2) En iyi LR ile seed 1, 2
for seed in [1, 2]:
    tag = f'gla_s{seed}_lr{BEST_LR}'
    run(['--model', 'gla', '--seed', str(seed), '--lr', BEST_LR, '--seq_length', '256',
         '--batch_size', '16', '--max_iters', '5000', '--log_tag', tag], tag)

print('\nGorev A tamam.')

## Görev B v2 — Uzun-bağlam karar deneyi (train@256 → eval@{256,1024,2048})

v1'de seq 1024'te eğitim denendi ve her kol `ln(vocab)` platosuna takıldı — bu RESULTS §3'teki *training-length cliff*; mimari değil optimizasyon sınırı. Doğru protokol repo bulgusuyla aynı: **kısa eğit, uzun koş.** 3 kol × 3 seed, seq 256, 5000 iter; sonra aynı ağırlıklarla 256/1024/2048'de val PPL.

**Önceden yazılı kriter:** eval 2048'de delta − additive > 2 SE ise hipotez doğrulanır; aksi halde reçete additive.

In [ ]:
# --- GOREV B v2: seq 256 egit -> {256,1024,2048} degerlendir ---
EVAL_LENS = '256,1024,2048'
ARMS = {
    'hfp_add':  ['--model', 'hfp', '--decay_mode', 'cubic_flux_chunked', '--write_rule', 'additive', '--key_feature_map', 'dpfp', '--lr', '5e-4'],
    'hfp_del':  ['--model', 'hfp', '--decay_mode', 'cubic_flux_chunked', '--write_rule', 'delta',    '--key_feature_map', 'dpfp', '--lr', '5e-4'],
    'gla':      ['--model', 'gla', '--lr', BEST_LR],
}
for arm, arm_args in ARMS.items():
    for seed in [0, 1, 2]:
        tag = f'TB_{arm}_s{seed}'
        run(arm_args + ['--seed', str(seed), '--seq_length', '256', '--batch_size', '16',
                        '--max_iters', '5000', '--eval_lens', EVAL_LENS,
                        '--log_tag', tag], tag)
print('\nGorev B v2 tamam.')

In [ ]:
# --- SONUC ANALIZI v2 (NaN korumali) ---
import numpy as np, math, csv

def best_val(tag):
    try:
        with open(f'{tag}_log.csv') as f:
            rows = list(csv.DictReader(f))
        vals = [float(r['val_loss']) for r in rows if not math.isnan(float(r['val_loss']))]
        return min(vals) if vals else None
    except Exception:
        return None

def lens_val(tag, elen):
    try:
        with open(f'{tag}_lens.csv') as f:
            for r in csv.DictReader(f):
                if int(r['eval_len']) == elen and not math.isnan(float(r['val_loss'])):
                    return float(r['val_loss'])
    except Exception:
        return None

def stats(vals):
    vals = [v for v in vals if v is not None]
    if len(vals) < 2: return None
    m = np.mean(vals); s = np.std(vals, ddof=1)
    return m, s, s/math.sqrt(len(vals)), len(vals)

print('='*78)
print('  GOREV A — GLA vs HFP (seq 256)')
print('='*78)
gla256 = stats([best_val(f'gla_s0_lr{BEST_LR}'), best_val(f'gla_s1_lr{BEST_LR}'), best_val(f'gla_s2_lr{BEST_LR}')])
HFP_REF = {'hfp cubic+additive+dpfp': ([5.2137,5.2156,5.2089],183.6),
           'hfp cubic+delta+dpfp':    ([5.2687,5.2667,5.2247],191.2)}
if gla256:
    print(f'  GLA (lr={BEST_LR}, n={gla256[3]}): val {gla256[0]:.4f} +/- {gla256[1]:.4f}  PPL {math.exp(gla256[0]):.1f}')
else:
    print('  GLA: gecerli sonuc yok (NaN/eksik) — KRITER 1 DEGERLENDIRILEMEZ')
for name,(vals,ppl_) in HFP_REF.items():
    st=stats(vals); print(f'  {name}: val {st[0]:.4f} +/- {st[1]:.4f}  PPL {ppl_}  [ablasyon ref]')
if gla256:
    hb=stats(HFP_REF['hfp cubic+additive+dpfp'][0])
    se_c=math.sqrt(gla256[2]**2+hb[2]**2)
    print(f'\n  KRITER 1: HFP-best - GLA = {hb[0]-gla256[0]:+.4f} (SE {se_c:.4f})')
    print('  ->','AILE-STANDARDI DOGRULANDI' if hb[0] <= gla256[0]+2*se_c else 'HFP >2 SE geride: iddia yapilmaz, durust deftere')

print()
print('='*78)
print('  GOREV B v2 — train@256 -> eval@{256,1024,2048}')
print('='*78)
res={}
for arm in ['hfp_add','hfp_del','gla']:
    res[arm]={}
    for elen in [256,1024,2048]:
        res[arm][elen]=stats([lens_val(f'TB_{arm}_s{s}',elen) for s in [0,1,2]])
    row=' | '.join(f'{elen}: '+(f'{st[0]:.4f} (PPL {math.exp(st[0]):.0f})' if (st:=res[arm][elen]) else 'yok') for elen in [256,1024,2048])
    print(f'  {arm:8s} {row}')
a,d=res['hfp_add'].get(2048),res['hfp_del'].get(2048)
if a and d:
    diff=a[0]-d[0]; se_c=math.sqrt(a[2]**2+d[2]**2)
    print(f'\n  KRITER 2 (eval 2048): additive - delta = {diff:+.4f} (SE {se_c:.4f})')
    if diff> 2*se_c: print('  -> delta kazandi (>2 SE): HIPOTEZ DOGRULANDI')
    elif diff<-2*se_c: print('  -> additive kazandi (>2 SE): hipotez reddedildi, recete cubic+additive+dpfp')
    else: print('  -> fark <2 SE: belirsiz; basitlik lehine additive, delta key-update nisinde')
else:
    print('  KRITER 2: eksik veri')
print('\nBitince: RESULTS.md guncellenecek (cift-§5 duzeltmesi dahil).')